# Numerical Algorithms Toolbox — Solvers You'll Use Everywhere

## Why this notebook exists

Across Modules 1–4 you called a **tridiagonal solver** for implicit diffusion (Module 1.07),
ran **Gauss-Seidel / SOR sweeps** inside SIMPLE for the pressure Poisson equation (Module 2.13),
and used **explicit Euler** for time-stepping flows (Module 2.14).
You used these algorithms — but never compared them side-by-side, never saw *why* SOR is faster
than Jacobi, and never connected RK4's "four stages" to anything geometric.

This notebook is the **engine-room reference** for every CFD code in this repository.

---

## Two families of algorithms

```
┌────────────────────────────────────────────────────────────┐
│  A · x = b   (algebraic linear system)                     │
│                                                            │
│  After implicit spatial discretisation you always end up   │
│  here — for pressure, temperature, species concentration.  │
│                                                            │
│  DIRECT  ──  Thomas algorithm, LU decomposition           │
│  ITERATIVE ─  Jacobi, Gauss-Seidel, SOR                   │
│  (Krylov methods — CG, GMRES — in Extra Module 12)        │
└────────────────────────────────────────────────────────────┘

┌────────────────────────────────────────────────────────────┐
│  du/dt = f(u, t)   (ODE / method-of-lines system)         │
│                                                            │
│  After spatial discretisation, time becomes a 1-D ODE.    │
│                                                            │
│  EXPLICIT  ─  Forward Euler, RK2, RK4                     │
│  IMPLICIT  ─  Backward Euler (leads back to A·x = b!)    │
└────────────────────────────────────────────────────────────┘
```

---

## Roadmap

| Section | Algorithms | Key idea |
|---------|-----------|----------|
| §1 | Why Ax = b everywhere | Implicit discretisation → linear system |
| §2 | Thomas, LU | Direct exact solvers |
| §3 | Jacobi, GS, SOR | Iterative solvers — spectral radius |
| §4 | Decision table | When to choose what |
| §5 | Euler, RK2, RK4 | ODE time integration — order and stability |
| §6 | Summary | One table: CFD task → recommended solver |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time
from scipy.linalg import lu_factor, lu_solve

## §1  Why A x = b Appears Everywhere in CFD

Every **implicit** spatial discretisation converts a PDE into a linear system.

**Example — 1-D implicit diffusion (BTCS, from Module 1.07):**

$$\frac{u_i^{n+1} - u_i^n}{\Delta t} = \alpha \frac{u_{i-1}^{n+1} - 2u_i^{n+1} + u_{i+1}^{n+1}}{\Delta x^2}$$

Rearranging for the unknowns $u^{n+1}$:

$$-r\,u_{i-1}^{n+1} + (1+2r)\,u_i^{n+1} - r\,u_{i+1}^{n+1} = u_i^n, \qquad r = \frac{\alpha\Delta t}{\Delta x^2}$$

Written for all interior nodes simultaneously:

$$\underbrace{\begin{bmatrix} 1{+}2r & -r & & \\ -r & 1{+}2r & -r & \\ & \ddots & \ddots & \ddots \end{bmatrix}}_{A\;\text{(tridiagonal)}} \underbrace{\begin{bmatrix} u_1^{n+1} \\ u_2^{n+1} \\ \vdots \end{bmatrix}}_{\mathbf{x}} = \underbrace{\begin{bmatrix} u_1^n + r u_0 \\ u_2^n \\ \vdots \end{bmatrix}}_{\mathbf{b}}$$

**Same pattern in 2-D:**

The SIMPLE pressure-correction equation (Module 2.13) gives a **2-D Poisson system** with a
5-diagonal sparse matrix.

### Properties of CFD matrices that matter for solver choice

| Property | 1-D diffusion | 2-D Poisson (SIMPLE) |
|----------|--------------|----------------------|
| Structure | Tridiagonal | Pentadiagonal (sparse) |
| Symmetry | Symmetric | Symmetric |
| Definiteness | SPD (r > 0) | SPD |
| Size (typical) | $N \sim 10^2$ | $N^2 \sim 10^4$–$10^6$ |
| Changes each time step? | Yes (via BCs) | Yes (velocity field changes) |

## §2  Direct Solvers — LU Decomposition & the Thomas Algorithm

### LU Decomposition

Factorise $A = LU$ (lower × upper triangular), then solve in two passes:

$$L\,\mathbf{y} = \mathbf{b} \quad(\text{forward substitution}), \qquad U\,\mathbf{x} = \mathbf{y} \quad(\text{back substitution})$$

Cost for a dense $n \times n$ matrix: $\mathcal{O}(n^3)$ for factorisation, $\mathcal{O}(n^2)$ per solve.
The factorisation can be **reused** if $A$ is the same and only $\mathbf{b}$ changes — very useful for
steady-state problems where the geometry is fixed.

### Thomas Algorithm — LU for Tridiagonals

When $A$ is tridiagonal (sub-diagonal $a$, main $b$, super-diagonal $c$), LU reduces to
$\mathcal{O}(n)$ operations — the **Thomas algorithm**:

**Forward sweep** (eliminate sub-diagonal):
$$c'_i = \frac{c_i}{b_i - a_i c'_{i-1}}, \qquad d'_i = \frac{d_i - a_i d'_{i-1}}{b_i - a_i c'_{i-1}}$$

**Back substitution:**
$$x_n = d'_n, \qquad x_i = d'_i - c'_i\, x_{i+1}$$

This is exactly what your 1-D implicit diffusion solver (Module 1.07) uses under the hood.

| Solver | Cost (factorisation) | Cost (per solve) | Memory |
|--------|---------------------|-----------------|--------|
| Dense LU | $\mathcal{O}(n^3)$ | $\mathcal{O}(n^2)$ | $\mathcal{O}(n^2)$ |
| Thomas algorithm | $\mathcal{O}(n)$ | $\mathcal{O}(n)$ | $\mathcal{O}(n)$ |

In [ ]:
def thomas_algorithm(a, b, c, d):
    '''Solve a tridiagonal system: a=sub, b=main, c=super, d=rhs. O(n).'''
    n = len(d)
    cp, dp = np.zeros(n), np.zeros(n)
    cp[0] = c[0] / b[0]
    dp[0] = d[0] / b[0]
    for i in range(1, n):
        m = b[i] - a[i] * cp[i-1]
        if i < n - 1:
            cp[i] = c[i] / m
        dp[i] = (d[i] - a[i] * dp[i-1]) / m
    x = np.zeros(n)
    x[-1] = dp[-1]
    for i in range(n-2, -1, -1):
        x[i] = dp[i] - cp[i] * x[i+1]
    return x

# Tridiagonal system from 1-D BTCS diffusion (Module 1.07 style)
# -r u_{i-1} + (1+2r) u_i - r u_{i+1} = u_i^n   with r = alpha*dt/dx^2
r = 0.4
results = {}
for n in [200, 1000]:
    a = -r   * np.ones(n)
    b = (1 + 2*r) * np.ones(n)
    c = -r   * np.ones(n)
    d = np.random.default_rng(0).random(n)   # previous time-step u^n

    # Build full dense matrix for LU comparison
    A = np.diag(b) + np.diag(a[1:], -1) + np.diag(c[:-1], 1)

    t0 = time.perf_counter()
    x_th = thomas_algorithm(a.copy(), b.copy(), c.copy(), d.copy())
    t_th = time.perf_counter() - t0

    t0 = time.perf_counter()
    lu, piv = lu_factor(A)
    x_lu = lu_solve((lu, piv), d)
    t_lu = time.perf_counter() - t0

    err = np.max(np.abs(x_th - x_lu))
    print(f'n={n:5d}: Thomas {t_th*1000:6.2f} ms | Dense LU {t_lu*1000:7.2f} ms '
          f'| speedup {t_lu/t_th:5.0f}x | max diff {err:.1e}')
    results[n] = (t_th, t_lu)

# Visualise the timing ratio
ns = list(results.keys())
speedup = [results[n][1] / results[n][0] for n in ns]
fig, ax = plt.subplots(figsize=(5, 3))
ax.bar([str(n) for n in ns], speedup, color='steelblue')
ax.set_xlabel('System size n')
ax.set_ylabel('Speedup of Thomas\nover dense LU')
ax.set_title('Thomas algorithm vs Dense LU\n(same tridiagonal system)')
plt.tight_layout()
plt.show()

### What to take away

- For a **tridiagonal** system of size $n = 1000$, Thomas is **~10–55× faster** than dense LU and
  uses $\mathcal{O}(n)$ memory instead of $\mathcal{O}(n^2)$.
- Dense LU cost is $\mathcal{O}(n^3)$ — it scales catastrophically: doubling $n$ increases cost 8×.
- Thomas cost is $\mathcal{O}(n)$ — doubling $n$ doubles the cost.

**When to use direct solvers:**

| Situation | Recommended |
|-----------|-------------|
| 1-D implicit discretisation (tridiagonal) | Thomas algorithm |
| Small dense system ($n < 500$), $A$ reused many times | LU (reuse factorisation) |
| 2-D/3-D large sparse system | Iterative (next section) |
| Banded system (ADI, block-tridiagonal) | Banded LU (`scipy.linalg.solve_banded`) |

For 2-D and 3-D CFD the sparse matrix has $\mathcal{O}(N^2)$ unknowns — direct LU becomes
prohibitively expensive, and sparse iterative methods take over.

## §3  Iterative Solvers — Jacobi, Gauss-Seidel, SOR

### Matrix splitting idea

Split $A = D + L + U$ (diagonal + strictly lower + strictly upper), then rewrite $Ax=b$ as:

$$x^{(k+1)} = \underbrace{-M^{-1}N}_{\text{iteration matrix }G}\, x^{(k)} + M^{-1}b$$

The iteration **converges** if and only if $\rho(G) < 1$, where $\rho$ is the **spectral radius**
(largest absolute eigenvalue). The smaller $\rho$, the faster the convergence.

### Three classical methods

**Jacobi** — update every component using *old* values only (fully parallel):

$$x_i^{(k+1)} = \frac{1}{a_{ii}}\left(b_i - \sum_{j \ne i} a_{ij}\, x_j^{(k)}\right)$$

**Gauss-Seidel (GS)** — use *updated* values as soon as they are available:

$$x_i^{(k+1)} = \frac{1}{a_{ii}}\left(b_i - \sum_{j < i} a_{ij}\, x_j^{(k+1)} - \sum_{j > i} a_{ij}\, x_j^{(k)}\right)$$

GS typically converges in **half the iterations** of Jacobi for the same problem (spectral radius
$\rho_{GS} = \rho_J^2$ for symmetric positive definite matrices from Poisson discretisations).

**SOR (Successive Over-Relaxation)** — blend the GS update with the current value:

$$x_i^{(k+1)} = (1-\omega)\,x_i^{(k)} + \omega\, x_i^{GS,(k+1)}, \qquad 0 < \omega < 2$$

For $\omega = 1$ this is plain GS.  For $\omega > 1$ (over-relaxation) convergence accelerates dramatically.

### Optimal SOR parameter for the 2-D Poisson equation

$$\boxed{\omega_{\text{opt}} = \frac{2}{1 + \sin(\pi h)}}, \qquad h = \text{grid spacing}$$

With $\omega_{\text{opt}}$, the spectral radius drops to $\rho_{SOR} = \omega_{\text{opt}} - 1$, giving
convergence that is **orders of magnitude faster** than GS.

| Method | $\rho$ (2-D Poisson, $h=0.05$) | Rough iterations to $10^{-6}$ |
|--------|-------------------------------|-------------------------------|
| Jacobi | $\cos(\pi h) \approx 0.988$ | $> 1000$ |
| Gauss-Seidel | $\rho_J^2 \approx 0.976$ | $\sim 500$ |
| SOR ($\omega_{\text{opt}}$) | $\omega_{\text{opt}}-1 \approx 0.730$ | $\sim 70$ |

In [ ]:
# 2-D Poisson MMS problem: Laplacian(p) = rhs on [0,1]^2 with Dirichlet p=0 on boundary
# Manufactured solution: p_exact = sin(pi x) sin(pi y)
# => rhs = -2 pi^2 sin(pi x) sin(pi y)
# Convergence is tracked via RESIDUAL NORM  ||A p - b||
# so discretisation error (O(h^2)) does not mask iterative convergence.

N = 21
h = 1.0 / (N - 1)
x = np.linspace(0, 1, N)
y = np.linspace(0, 1, N)
X, Y = np.meshgrid(x, y, indexing='ij')
rhs = -2 * np.pi**2 * np.sin(np.pi*X) * np.sin(np.pi*Y)   # known RHS

def residual_rms(p, rhs, h):
    '''RMS of Ap - b over interior nodes.'''
    lap = (p[2:,1:-1] + p[:-2,1:-1] + p[1:-1,2:] + p[1:-1,:-2] - 4*p[1:-1,1:-1]) / h**2
    return np.sqrt(np.mean((rhs[1:-1,1:-1] - lap)**2))

def jacobi_solve(p, rhs, h, n_iter):
    history = [residual_rms(p, rhs, h)]
    for _ in range(n_iter):
        pn = p.copy()
        pn[1:-1,1:-1] = 0.25*(p[2:,1:-1] + p[:-2,1:-1]
                              + p[1:-1,2:] + p[1:-1,:-2]
                              - h**2 * rhs[1:-1,1:-1])
        p = pn
        history.append(residual_rms(p, rhs, h))
    return p, history

def sor_solve(p, rhs, h, n_iter, omega=1.0):
    '''omega=1 gives plain Gauss-Seidel; omega > 1 gives SOR.'''
    history = [residual_rms(p, rhs, h)]
    n = p.shape[0]
    for _ in range(n_iter):
        for i in range(1, n-1):
            for j in range(1, n-1):
                p_gs = 0.25*(p[i+1,j] + p[i-1,j] + p[i,j+1] + p[i,j-1]
                             - h**2 * rhs[i,j])
                p[i,j] = (1 - omega)*p[i,j] + omega*p_gs
        history.append(residual_rms(p, rhs, h))
    return p, history

omega_opt = 2.0 / (1 + np.sin(np.pi * h))
print(f'Grid: {N}x{N}, h = {h:.4f},  omega_opt = {omega_opt:.4f}')

N_ITER = 150
p0 = np.zeros((N, N))
_, h_jac = jacobi_solve(p0.copy(), rhs, h, N_ITER)
_, h_gs  = sor_solve(p0.copy(), rhs, h, N_ITER, omega=1.0)
_, h_sor = sor_solve(p0.copy(), rhs, h, N_ITER, omega=omega_opt)

for name, hist in [('Jacobi', h_jac), ('Gauss-Seidel', h_gs), ('SOR', h_sor)]:
    arr = np.array(hist)
    idx = next((i for i, v in enumerate(arr) if v < 1e-6), None)
    print(f'{name:14s}: residual after {N_ITER} iters = {arr[-1]:.2e} '
          f'| iters to 1e-6: {idx if idx is not None else ">"+str(N_ITER)}')

fig, ax = plt.subplots(figsize=(7, 5))
ax.semilogy(h_jac, lw=2, label='Jacobi')
ax.semilogy(h_gs,  lw=2, label='Gauss-Seidel')
ax.semilogy(h_sor, lw=2, label=f'SOR  ($\\omega$ = {omega_opt:.3f})')
ax.axhline(1e-6, color='gray', ls=':', lw=1, label='tolerance 1e-6')
ax.set_xlabel('Iteration (sweep)', fontsize=11)
ax.set_ylabel('Residual RMS  $\\|Ap - b\\|$', fontsize=11)
ax.set_title('Convergence: Jacobi vs Gauss-Seidel vs SOR\n(2-D Poisson, same problem)', fontsize=11)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

### What to take away

The plot makes the spectral-radius theory concrete:

- **Jacobi** barely moves — residual drops from ~10 to ~1.6 in 150 sweeps (spectral radius 0.988
  means each sweep removes only 1.2% of the error).
- **Gauss-Seidel** converges roughly 4× faster than Jacobi for this grid.
- **SOR with $\omega_{\text{opt}}$** reaches tolerance $10^{-6}$ in **67 sweeps** while Jacobi is still
  at residual ~1.6.  One order-of-magnitude difference in iteration count.

### Connection to what you already built

The **SIMPLE pressure-correction loop** in Module 2.13 is essentially a SOR sweep:
```
p_new = p + omega * p_prime   (pressure-correction update in SIMPLE)
```
The under-relaxation factor used there is exactly this $\omega$ — for 2-D cavity flow the
typical value 1.0–1.7 comes from the same SOR theory.

**Jacobi** is used in parallel CFD codes because each new value depends only on old values
(no sequential dependency), making it trivially parallelisable — but it needs twice as many
iterations as GS.

> **Next step:** When the matrix becomes very large (3-D, refined mesh), even SOR becomes slow.
> Krylov methods (CG, GMRES) replace it in production codes — see Extra Module 12.

## §4  Decision Table — Which Solver for Which Problem?

| Problem | Matrix structure | Recommended solver | Why |
|---------|-----------------|-------------------|-----|
| 1-D implicit diffusion / advection | Tridiagonal | **Thomas algorithm** | $\mathcal{O}(n)$, exact, trivial to code |
| 2-D/3-D steady Poisson (small grid) | Pentadiagonal sparse | **SOR** with $\omega_{\text{opt}}$ | Much faster convergence than Jacobi/GS |
| 2-D/3-D steady Poisson (large grid) | Sparse SPD | **Conjugate Gradient (CG)** + preconditioner | Optimal Krylov method for SPD |
| Non-symmetric (momentum equation) | Sparse non-symmetric | **GMRES** or **BiCGSTAB** | Krylov methods for non-SPD |
| Dense system, $n < 500$, $A$ fixed | Dense | **LU** (cache factorisation) | $\mathcal{O}(n^2)$ per solve after one $\mathcal{O}(n^3)$ factorisation |
| Parallel solver | Any | **Jacobi** or **domain-decomposed SOR** | Jacobi update only needs neighbours' *old* values |

### A note on Krylov methods

CG, GMRES, and BiCGSTAB are **not** covered in this notebook (they deserve their own depth —
see **Extra Module 12**).  The key idea: instead of a fixed update formula (Jacobi/GS/SOR), Krylov
methods build an optimal solution in the subspace spanned by $\{r, Ar, A^2r, \ldots\}$, converging
in at most $n$ steps for a symmetric system.  In practice they converge in far fewer.

## §5  Time Integration — ODE Solvers for $du/dt = f(u,t)$

After spatial discretisation of an unsteady PDE (e.g., the 2-D advection-diffusion equation),
we get a **system of ODEs** in time — this is the "method of lines."  The remaining task is to
march those ODEs forward in time.

### Forward (Explicit) Euler

$$\boxed{u^{n+1} = u^n + \Delta t\, f(u^n, t^n)}$$

- First-order accurate: $\text{error} = \mathcal{O}(\Delta t)$
- Conditionally stable: requires $\lambda \Delta t$ to lie in the **stability region**
  (the unit circle centred at $-1$ in the complex plane) — this is the **CFL condition** in CFD.

### Backward (Implicit) Euler

$$u^{n+1} = u^n + \Delta t\, f(u^{n+1}, t^{n+1})$$

- First-order accurate — same order as forward Euler but **unconditionally stable**.
- Requires solving $A\,u^{n+1} = b$ at every time step — ties directly to §2 and §3.

### Runge-Kutta Methods (Explicit)

The idea: evaluate $f$ at **multiple stages** within $[t^n, t^{n+1}]$ and combine them
to cancel low-order error terms.

**RK2 (Heun / midpoint)** — 2 stages, 2nd-order:
$$k_1 = f(t^n, u^n), \qquad k_2 = f(t^n + \Delta t,\; u^n + \Delta t\, k_1)$$
$$u^{n+1} = u^n + \frac{\Delta t}{2}(k_1 + k_2)$$

**RK4 (classical)** — 4 stages, 4th-order:
$$k_1 = f(t^n,\; u^n), \qquad k_2 = f\!\left(t^n + \tfrac{\Delta t}{2},\; u^n + \tfrac{\Delta t}{2}k_1\right)$$
$$k_3 = f\!\left(t^n + \tfrac{\Delta t}{2},\; u^n + \tfrac{\Delta t}{2}k_2\right), \qquad k_4 = f(t^n + \Delta t,\; u^n + \Delta t\, k_3)$$
$$u^{n+1} = u^n + \frac{\Delta t}{6}(k_1 + 2k_2 + 2k_3 + k_4)$$

**Order of accuracy** means: halve $\Delta t$ → error shrinks by $2^p$ where $p$ is the order.

| Method | Order $p$ | Stages | Stability limit (real axis) |
|--------|----------|--------|----------------------------|
| Forward Euler | 1 | 1 | $\lambda\Delta t \in [-2, 0]$ |
| RK2 | 2 | 2 | $\lambda\Delta t \in [-2, 0]$ |
| RK4 | 4 | 4 | $\lambda\Delta t \in [-2.83, 0]$ |
| Backward Euler | 1 | 1 (implicit) | entire left half-plane |

### Stiff problems

If $f$ contains both fast and slow modes (e.g., diffusion with large $\alpha$), explicit methods
require tiny $\Delta t$ for stability even when accuracy allows larger steps.  Such problems are
called **stiff** — they demand implicit schemes.

In CFD: the **viscous term** (diffusion) is stiff at fine grids; the **advective term** is stiff
at high velocities.  Many codes treat diffusion implicitly and advection explicitly — a
**"semi-implicit"** or **IMEX** approach.

In [ ]:
# ── Part A: Convergence order ─────────────────────────────────────────────────
# Test problem:  du/dt = -u,  u(0) = 1,  exact solution: u(t) = exp(-t)
# Integrate to t=1 with various step sizes, measure error.

def euler_step(f, u, t, dt):
    return u + dt * f(t, u)

def rk2_step(f, u, t, dt):
    k1 = f(t, u)
    k2 = f(t + dt, u + dt * k1)
    return u + dt / 2 * (k1 + k2)

def rk4_step(f, u, t, dt):
    k1 = f(t, u)
    k2 = f(t + dt/2, u + dt/2 * k1)
    k3 = f(t + dt/2, u + dt/2 * k2)
    k4 = f(t + dt,   u + dt   * k3)
    return u + dt / 6 * (k1 + 2*k2 + 2*k3 + k4)

f_decay  = lambda t, u: -u
u_exact  = np.exp(-1.0)

# Powers of 2 so that 1.0/dt is always an exact integer
dts = np.array([0.5, 0.25, 0.125, 0.0625, 0.03125, 0.015625])

methods = [('Forward Euler', euler_step, 'o-', 'C0'),
           ('RK2',           rk2_step,   's-', 'C1'),
           ('RK4',           rk4_step,   '^-', 'C2')]

errors = {}
for name, step, *_ in methods:
    errs = []
    for dt in dts:
        u, t = 1.0, 0.0
        for _ in range(int(round(1.0 / dt))):
            u = step(f_decay, u, t, dt)
            t += dt
        errs.append(abs(u - u_exact))
    errors[name] = np.array(errs)
    obs = np.log(errs[-2] / errs[-1]) / np.log(dts[-2] / dts[-1])
    print(f'{name:14s}: observed order = {obs:.2f}')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ax = axes[0]
for name, _, marker, color in methods:
    ax.loglog(dts, errors[name], marker, color=color, lw=2, label=name)
# Reference slopes anchored to each curve's midpoint
mid = len(dts) // 2
ax.loglog(dts, errors['Forward Euler'][mid] / dts[mid]    * dts,    'k--', alpha=0.35, lw=1.2, label='slope 1')
ax.loglog(dts, errors['RK4'][mid]           / dts[mid]**4 * dts**4, 'k:',  alpha=0.35, lw=1.2, label='slope 4')
ax.invert_xaxis()
ax.set_xlabel(r'Step size $\Delta t$', fontsize=11)
ax.set_ylabel('Error at t = 1', fontsize=11)
ax.set_title('Convergence order\n(Euler=1, RK2=2, RK4=4)', fontsize=11)
ax.legend(fontsize=9)

# ── Part B: Absolute stability regions ────────────────────────────────────────
ax = axes[1]
xg = np.linspace(-4.5, 1.5, 600)
yg = np.linspace(-3.2, 3.2, 600)
Xg, Yg = np.meshgrid(xg, yg)
Z = Xg + 1j*Yg

stab = [('Forward Euler', np.abs(1 + Z),                                'C0'),
        ('RK2',           np.abs(1 + Z + Z**2/2),                       'C1'),
        ('RK4',           np.abs(1 + Z + Z**2/2 + Z**3/6 + Z**4/24),   'C2')]

for label, R, color in stab:
    ax.contourf(Xg, Yg, R, levels=[0, 1], colors=[color], alpha=0.22)
    ax.contour( Xg, Yg, R, levels=[1],    colors=[color], linewidths=2)
    ax.plot([], [], color=color, lw=3, label=label)

ax.axhline(0, color='k', lw=0.7)
ax.axvline(0, color='k', lw=0.7)
ax.set_xlabel(r'Re($\lambda\Delta t$)', fontsize=11)
ax.set_ylabel(r'Im($\lambda\Delta t$)', fontsize=11)
ax.set_title('Absolute stability regions\n(shaded = stable)', fontsize=11)
ax.legend(fontsize=9)
ax.set_aspect('equal')
ax.annotate('CFL constraint:\nstay inside your\nmethod\'s region',
            xy=(-1,0), xytext=(0.3, 1.5),
            arrowprops=dict(arrowstyle='->', color='black'),
            fontsize=8.5)

plt.tight_layout()
plt.show()

### What to take away

**Convergence order plot (left):**

- Halve $\Delta t$ and the Forward Euler error halves (slope 1).
- RK2 gives **4× smaller** error for the same halving of $\Delta t$ (slope 2).
- RK4 gives **16× smaller** error (slope 4) — for smooth problems it is extremely accurate
  with moderate $\Delta t$.

At $\Delta t = 0.015625$ RK4 has ~$10^{-10}$ error vs Euler's ~$10^{-3}$ — **7 orders of
magnitude** difference for the same computational cost (64 steps each).

**Stability regions (right):**

- All three explicit methods are stable **only inside their shaded region** in the
  $\lambda\Delta t$ complex plane.
- For a diffusion-dominated problem $\lambda \approx -\alpha/\Delta x^2$ (real, negative).
  The CFL constraint is $|\lambda|\Delta t \leq C$ where $C$ depends on the method.
- RK4's wider region (extends to $-2.83$ on the real axis vs Euler's $-2$) allows a
  **larger time step** for the same spatial resolution.
- **Implicit Euler** has the entire left half-plane as its stability region — no CFL
  constraint, but you must solve $Ax=b$ at every step.

### Which method to pick in CFD?

| Flow type | Common choice | Reason |
|-----------|--------------|--------|
| Incompressible, steady state | Implicit Euler (or steady iterations) | No CFL limit |
| Incompressible, unsteady | PISO/PIMPLE with implicit Euler | Stable at large $\Delta t$ |
| DNS / LES (turbulence) | RK4 or 3rd-order RK | Accuracy matters; $\Delta t$ is small anyway |
| Compressible / shock capturing | RK3 (Shu-Osher) or RK4 | Explicit; shocks need small $\Delta t$ |

## §6  Grand Summary — CFD Task → Recommended Solver

| CFD task (where in curriculum) | System type | Recommended algorithm | Notes |
|-------------------------------|-------------|----------------------|-------|
| 1-D implicit diffusion (Module 1.07) | Tridiagonal | **Thomas algorithm** | $\mathcal{O}(n)$, exact |
| SIMPLE pressure correction (Module 2.13) | Sparse 2-D Poisson | **SOR** ($\omega_{\text{opt}}$) or **CG** | SOR for small grids; CG for large |
| Momentum equation (Module 2.13) | Sparse non-symmetric | **Jacobi / GS** sweeps or **BiCGSTAB** | Under-relaxation controls stability |
| Lid-driven cavity time marching | ODE system | **Implicit Euler** (steady) | No CFL limit; inner solve needed |
| Unsteady DNS / LES (Extra Module 01) | ODE system | **RK4** or **RK3** | High-order time accuracy needed |
| Turbulence transport equations (k-ε, Module 3.18) | Sparse | **BiCGSTAB** or **GS sweeps** | Non-symmetric, stiff at wall |
| Tridiagonal block systems (ADI methods) | Block tridiagonal | **Block Thomas** | Direct, $\mathcal{O}(n)$ per direction |
| Large 3-D problems, structured mesh | Sparse SPD | **CG + ILU preconditioner** | → Extra Module 12 (Krylov) |
| Large 3-D problems, unstructured mesh | Sparse non-SPD | **GMRES + ILU preconditioner** | → Extra Module 12 (Krylov) |

### Spectral radius at a glance (2-D Poisson, $h = 0.05$)

| Method | $\rho$ | Relative iterations to converge |
|--------|--------|--------------------------------|
| Jacobi | 0.988 | 1× (baseline) |
| Gauss-Seidel | 0.976 | ~0.5× |
| SOR ($\omega_{\text{opt}}$) | 0.730 | ~0.05× |
| Conjugate Gradient | $\sim\sqrt{\kappa}$ scaling | $\ll 0.01\times$ (with good preconditioner) |

### Time-integrator at a glance

| Method | Order | Stable $\lambda\Delta t$ (real) | Implicit? |
|--------|-------|--------------------------------|-----------|
| Forward Euler | 1 | $[-2,\;0]$ | No |
| RK2 | 2 | $[-2,\;0]$ | No |
| RK4 | 4 | $[-2.83,\;0]$ | No |
| Backward Euler | 1 | entire $\text{Re}<0$ | Yes |
| Crank-Nicolson | 2 | entire $\text{Re}<0$ | Yes |

## Exercise — Predict First, Then Verify

### Part 1 — Iterative convergence (Jacobi/GS/SOR)

Without running any code, answer:

**(a)** If you double the grid to $N = 41$ ($h = 0.025$), does $\omega_{\text{opt}}$ go **up or down**?
Compute it analytically.

**(b)** For the finer grid, how many SOR iterations do you expect to reach residual $< 10^{-6}$?
Use $\rho_{SOR} = \omega_{\text{opt}} - 1$ and the formula
$k \approx \log(10^{-6} / r_0) / \log(\rho)$.

**(c)** Re-run the iterative demo cell with `N = 41` and check whether SOR matches your prediction.

---

### Part 2 — Time integration order

**(a)** Using RK2 with $\Delta t = 0.25$ on the test ODE $du/dt = -u$, compute the solution at
$t = 1$ by hand (two steps: $t=0\to0.25$, then a different $k_1, k_2$ for $t=0.25\to0.5$ …
wait, use just $\Delta t = 0.5$, one step: $t=0\to0.5$, then another $t=0.5\to1$).
Compare with the exact value $e^{-1} \approx 0.3679$.

**(b)** Without running code: if you switch from RK2 to RK4 and use the same $\Delta t = 0.25$,
roughly how many orders of magnitude smaller do you expect the error to be?
*(Hint: $\text{error}_{RK4} \approx \text{error}_{RK2} \times (\Delta t)^{4-2}$)*

**(c)** Run the time-integration demo cell with an additional `dts` value of $0.03125$ and
verify your prediction from (b).

---

### Part 3 — Connecting solvers to a real CFD code

Look at your `exercise/lid_driven_cavity.ipynb`.

**(a)** Find the pressure-correction loop. Which iterative solver is being used, and what is the
relaxation parameter $\omega$?

**(b)** Re-run the cavity solver with $\omega = 1.0$ (plain GS) vs your current value.
How many SIMPLE iterations does each need to reach the same residual?

**(c)** Try $\omega = \omega_{\text{opt}}$ from the formula above (with $h$ from your cavity grid).
Does it converge faster or cause divergence? Why might $\omega_{\text{opt}}$ not be exactly
optimal for the coupled NS system?